# CrimeScope ML — 03 (UK). Panel + Features + Demographics

Joins `uk_crime_monthly_lsoa` / `_msoa` with ONS Census 2021 + IMD 2019 (England)
+ WIMD 2019 (Wales), builds a dense panel (every LSOA × every month with zeros),
and engineers ~50 features mirroring `crimescope/ml/train.py` so the LightGBM
recipe transfers directly.

**Tables written:**
- `uk_lsoa_demographics` — population, age, deprivation per LSOA
- `uk_msoa_demographics` — same, per MSOA (population-weighted aggregate)
- `uk_lsoa_features` — feature table, primary training input
- `uk_msoa_features` — feature table, MSOA rollup

In [ ]:
spark.sql("USE CATALOG varanasi")
spark.sql("USE SCHEMA default")

---
## 1. ONS Census 2021 (Nomis API)

Pull total population (`TS001`) and population-by-age (`TS007A`) at LSOA level
for England & Wales. Nomis exposes Census 2021 as table `NM_2021_1` (TS001).

In [ ]:
import io
import urllib.request
import urllib.parse
import pandas as pd

VOLUME_RAW = "/Volumes/varanasi/default/ml_data_uk/raw"

# Nomis bulk CSV download — all E&W LSOAs in one shot
# TS001 is total population by LSOA 2021
NOMIS_TS001 = (
    "https://www.nomisweb.co.uk/api/v01/dataset/NM_2021_1.bulk.csv?"
    "date=latest&geography=TYPE151&measures=20100"
)

def fetch_csv(url: str, dst: str) -> str:
    import os
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(dst) and os.path.getsize(dst) > 50_000:
        return dst
    req = urllib.request.Request(url, headers={"User-Agent": "CrimeScope-UK/1.0"})
    with urllib.request.urlopen(req, timeout=300) as resp, open(dst, "wb") as f:
        while True:
            buf = resp.read(1 << 20)
            if not buf:
                break
            f.write(buf)
    return dst


ts001 = fetch_csv(NOMIS_TS001, f"{VOLUME_RAW}/census/ts001_population.csv")
pop_pdf = pd.read_csv(ts001)
# Nomis returns LSOA code in either GEOGRAPHY_CODE or geography_code
code_col = next(c for c in pop_pdf.columns if c.lower() == "geography code")
# OBS_VALUE for total population
val_col = next(c for c in pop_pdf.columns if "OBS_VALUE" in c.upper() or c.lower() == "observation")
pop_pdf = pop_pdf[[code_col, val_col]].rename(columns={code_col: "lsoa_code", val_col: "total_pop"})
pop_pdf = pop_pdf[pop_pdf["lsoa_code"].astype(str).str.startswith(("E0", "W0"))]
pop_pdf["total_pop"] = pd.to_numeric(pop_pdf["total_pop"], errors="coerce")
print(f"Population rows: {len(pop_pdf):,}")
display(pop_pdf.head())

---
## 2. English IMD 2019 (LSOA, all 7 domains)

In [ ]:
# MHCLG English IMD 2019 — File 7 has all sub-domain ranks + scores by LSOA
IMD_EN_URL = (
    "https://assets.publishing.service.gov.uk/media/"
    "5d8b3a2540f0b609909b5908/"
    "File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3.xlsx"
)
imd_en_path = fetch_csv(IMD_EN_URL, f"{VOLUME_RAW}/imd/imd_en_2019.xlsx")

imd_en = pd.read_excel(imd_en_path, sheet_name="IoD2019 Scores")
# The first column is LSOA code 2011 — Census 2021 LSOAs are mostly identical, with ~3% changes
imd_en.columns = [str(c).strip() for c in imd_en.columns]
imd_en = imd_en.rename(columns={
    "LSOA code (2011)": "lsoa_code_2011",
    "Index of Multiple Deprivation (IMD) Score": "imd_score",
    "Income Score (rate)": "imd_income",
    "Employment Score (rate)": "imd_employment",
    "Education, Skills and Training Score": "imd_education",
    "Health Deprivation and Disability Score": "imd_health",
    "Crime Score": "imd_crime",
    "Barriers to Housing and Services Score": "imd_housing",
    "Living Environment Score": "imd_environment",
})
imd_en_keep = [
    "lsoa_code_2011", "imd_score", "imd_income", "imd_employment",
    "imd_education", "imd_health", "imd_crime", "imd_housing", "imd_environment",
]
imd_en = imd_en[imd_en_keep].copy()
# National decile (1 = most deprived)
imd_en["imd_decile"] = pd.qcut(imd_en["imd_score"].rank(method="first"), 10, labels=range(10, 0, -1)).astype(int)
print(f"English IMD rows: {len(imd_en):,}")

---
## 3. Welsh IMD 2019 (LSOA-level)

Wales publishes WIMD 2019 separately on StatsWales (different methodology). We
normalize each domain to a 0–100 percentile within Wales and assign deciles, so
downstream feature names match `imd_*`.

In [ ]:
# StatsWales WIMD 2019 LSOA scores (CSV via the open data API)
WIMD_URL = "https://statswales.gov.wales/Download/File?fileId=605"
wimd_path = fetch_csv(WIMD_URL, f"{VOLUME_RAW}/imd/wimd_2019.csv")
try:
    wimd = pd.read_csv(wimd_path, encoding="utf-8")
except UnicodeDecodeError:
    wimd = pd.read_csv(wimd_path, encoding="latin-1")
# WIMD ranks domains; convert to 0-100 percentiles to match scale
rank_cols = [c for c in wimd.columns if "Rank" in c]
if "LSOA Code" in wimd.columns:
    wimd = wimd.rename(columns={"LSOA Code": "lsoa_code_2011"})

wimd_norm = pd.DataFrame({"lsoa_code_2011": wimd["lsoa_code_2011"]}) if "lsoa_code_2011" in wimd.columns else pd.DataFrame()
# Bring in main rank if present
for cand, target in [
    ("WIMD 2019 Rank", "imd_score"),
    ("Income Rank", "imd_income"),
    ("Employment Rank", "imd_employment"),
    ("Education Rank", "imd_education"),
    ("Health Rank", "imd_health"),
    ("Community Safety Rank", "imd_crime"),
    ("Access to Services Rank", "imd_housing"),
    ("Physical Environment Rank", "imd_environment"),
]:
    if cand in wimd.columns and "lsoa_code_2011" in wimd.columns:
        # Convert rank to a 0-100 score (high = more deprived to match English convention)
        ranks = pd.to_numeric(wimd[cand], errors="coerce")
        wimd_norm[target] = (1 - (ranks - 1) / max(ranks.max() - 1, 1)) * 100
if not wimd_norm.empty:
    wimd_norm["imd_decile"] = pd.qcut(
        wimd_norm["imd_score"].rank(method="first"), 10, labels=range(10, 0, -1)
    ).astype(int)
    print(f"Welsh IMD rows: {len(wimd_norm):,}")
else:
    print("Welsh IMD parse fell back to empty (CSV schema unexpected); will rely on English IMD only")
    wimd_norm = pd.DataFrame(columns=imd_en.columns)

---
## 4. Build `uk_lsoa_demographics` (and a population-weighted MSOA rollup)

In [ ]:
imd = pd.concat([imd_en, wimd_norm], ignore_index=True).drop_duplicates("lsoa_code_2011")
# Census 2021 codes match 2011 LSOA codes for >97% of E&W (boundary changes in 2021)
demo = pop_pdf.merge(imd, left_on="lsoa_code", right_on="lsoa_code_2011", how="left")
demo = demo.drop(columns=["lsoa_code_2011"], errors="ignore")
# Sensible imputation: fill missing IMD with national median so models don't trip on NaNs
for col in ["imd_score", "imd_income", "imd_employment", "imd_education",
            "imd_health", "imd_crime", "imd_housing", "imd_environment"]:
    if col in demo.columns:
        demo[col] = demo[col].fillna(demo[col].median())
if "imd_decile" in demo.columns:
    demo["imd_decile"] = demo["imd_decile"].fillna(5).astype(int)

spark.createDataFrame(demo).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("varanasi.default.uk_lsoa_demographics")
spark.sql("ALTER TABLE varanasi.default.uk_lsoa_demographics SET TBLPROPERTIES ('comment' = 'Per-LSOA demographics: ONS Census 2021 population + IMD/WIMD 2019.')")
print(f"uk_lsoa_demographics: {len(demo):,} rows")

In [ ]:
# MSOA demographics: population-weighted average over child LSOAs
spark.sql("""
  CREATE OR REPLACE TABLE varanasi.default.uk_msoa_demographics
  COMMENT 'Per-MSOA demographics: pop-weighted aggregate of LSOA demographics.'
  AS
  SELECT
    l.msoa_code,
    SUM(d.total_pop) AS total_pop,
    SUM(d.total_pop * d.imd_score)        / NULLIF(SUM(d.total_pop), 0) AS imd_score,
    SUM(d.total_pop * d.imd_income)       / NULLIF(SUM(d.total_pop), 0) AS imd_income,
    SUM(d.total_pop * d.imd_employment)   / NULLIF(SUM(d.total_pop), 0) AS imd_employment,
    SUM(d.total_pop * d.imd_education)    / NULLIF(SUM(d.total_pop), 0) AS imd_education,
    SUM(d.total_pop * d.imd_health)       / NULLIF(SUM(d.total_pop), 0) AS imd_health,
    SUM(d.total_pop * d.imd_crime)        / NULLIF(SUM(d.total_pop), 0) AS imd_crime,
    SUM(d.total_pop * d.imd_housing)      / NULLIF(SUM(d.total_pop), 0) AS imd_housing,
    SUM(d.total_pop * d.imd_environment)  / NULLIF(SUM(d.total_pop), 0) AS imd_environment,
    CAST(ROUND(SUM(d.total_pop * d.imd_decile) / NULLIF(SUM(d.total_pop), 0)) AS INT) AS imd_decile
  FROM varanasi.default.uk_lsoa_demographics d
  JOIN varanasi.default.uk_lsoa_to_msoa_lookup l
    ON d.lsoa_code = l.lsoa_code
  GROUP BY l.msoa_code
""")

---
## 5. Dense panel + features (LSOA)

Mirrors `crimescope/ml/train.py` `engineer_features`: lags 1/2/3/6/12, rolling
3/6/12 (mean/std/max/min), MoM/YoY, violent/property sub-features, calendar
encoding, demographics, force-wide and MSOA-wide context.

In [ ]:
import math
import numpy as np

monthly = spark.table("varanasi.default.uk_crime_monthly_lsoa").toPandas()
demo_pdf = spark.table("varanasi.default.uk_lsoa_demographics").toPandas()
lookup_pdf = spark.table("varanasi.default.uk_lsoa_to_msoa_lookup").toPandas()

all_lsoa = demo_pdf["lsoa_code"].unique()
all_months = pd.date_range(monthly["month_start"].min(), monthly["month_start"].max(), freq="MS")

idx = pd.MultiIndex.from_product([all_lsoa, all_months], names=["lsoa_code", "month_start"])
panel = pd.DataFrame(index=idx).reset_index()
panel["month_start"] = panel["month_start"].dt.date
panel = panel.merge(monthly, on=["lsoa_code", "month_start"], how="left")
for c in ["incident_count", "violent_count", "property_count"]:
    panel[c] = panel[c].fillna(0).astype(int)
panel = panel.merge(lookup_pdf[["lsoa_code", "msoa_code", "lad_code"]], on="lsoa_code", how="left")
panel = panel.sort_values(["lsoa_code", "month_start"]).reset_index(drop=True)

# Labels (next-30-day = next month)
g = panel.groupby("lsoa_code")
panel["y_next_30d_count"] = g["incident_count"].shift(-1)
panel["y_next_30d_violent"] = g["violent_count"].shift(-1)
panel["y_next_30d_property"] = g["property_count"].shift(-1)
panel["y_incidents_12m"] = g["incident_count"].transform(lambda x: x.rolling(12, min_periods=1).sum())
panel["y_violent_12m"] = g["violent_count"].transform(lambda x: x.rolling(12, min_periods=1).sum())
panel["y_property_12m"] = g["property_count"].transform(lambda x: x.rolling(12, min_periods=1).sum())
panel = panel.dropna(subset=["y_next_30d_count"])
for c in ["y_next_30d_count", "y_next_30d_violent", "y_next_30d_property"]:
    panel[c] = panel[c].astype(int)

# Maturity buffer — drop the most recent 2 months
panel["month_dt"] = pd.to_datetime(panel["month_start"])
cutoff = panel["month_dt"].max() - pd.DateOffset(months=2)
panel = panel[panel["month_dt"] <= cutoff].drop(columns=["month_dt"])
print(f"Panel: {len(panel):,} rows, {panel['lsoa_code'].nunique()} LSOAs, {panel['month_start'].nunique()} months")

In [ ]:
df = panel.copy()
df = df.sort_values(["lsoa_code", "month_start"]).reset_index(drop=True)
g  = df.groupby("lsoa_code")["incident_count"]
gv = df.groupby("lsoa_code")["violent_count"]
gp = df.groupby("lsoa_code")["property_count"]

for lag in [1, 2, 3, 6, 12]:
    df[f"lag_{lag}m_count"] = g.shift(lag)
for period in [3, 6, 12]:
    df[f"rolling_mean_{period}m"] = g.transform(lambda x: x.shift(1).rolling(period, min_periods=1).mean())
    if period >= 6:
        df[f"rolling_std_{period}m"] = g.transform(lambda x: x.shift(1).rolling(period, min_periods=1).std())
        df[f"rolling_max_{period}m"] = g.transform(lambda x: x.shift(1).rolling(period, min_periods=1).max())
        df[f"rolling_min_{period}m"] = g.transform(lambda x: x.shift(1).rolling(period, min_periods=1).min())
df["mom_change"] = df["incident_count"] - g.shift(1)

for crime_type, grp in [("violent", gv), ("property", gp)]:
    for lag in [1, 3, 6]:
        df[f"{crime_type}_lag_{lag}m"] = grp.shift(lag)
    for period in [3, 6, 12]:
        df[f"{crime_type}_rolling_{period}m"] = grp.transform(
            lambda x: x.shift(1).rolling(period, min_periods=1).mean()
        )

df["violent_ratio"] = np.where(df["incident_count"] > 0, df["violent_count"] / df["incident_count"], 0.0)
df["violent_ratio_6m"] = df.groupby("lsoa_code")["violent_ratio"].transform(
    lambda x: x.shift(1).rolling(6, min_periods=1).mean()
)

# Calendar
dt = pd.to_datetime(df["month_start"])
df["month_of_year"] = dt.dt.month
df["month_sin"] = np.sin(2 * math.pi * df["month_of_year"] / 12)
df["month_cos"] = np.cos(2 * math.pi * df["month_of_year"] / 12)
df["year"] = dt.dt.year
df["same_month_last_year"] = g.shift(12)
df["yoy_change"] = np.where(df["same_month_last_year"].notna(), df["incident_count"] - df["same_month_last_year"], np.nan)

# Demographics
df = df.merge(demo_pdf, on="lsoa_code", how="left")
df["log_pop"] = np.where(df["total_pop"] > 0, np.log(df["total_pop"]), 0.0)

rm12 = df["rolling_mean_12m"].fillna(0)
df["crime_rate_per_1k"] = np.where((df["total_pop"] > 0) & (rm12 > 0), rm12 / (df["total_pop"] / 1000.0), 0.0)
df["imd_x_crime"] = df["imd_score"].fillna(0) * rm12

# MSOA-wide context
msoa_avg = df.groupby(["msoa_code", "month_start"])["incident_count"].transform("mean")
df["msoa_avg_crime"] = msoa_avg.fillna(df["incident_count"])
df["lsoa_vs_msoa_avg"] = np.where(df["msoa_avg_crime"] > 0, df["incident_count"] / df["msoa_avg_crime"], 1.0)

# Force-wide context (LAD as proxy — the per-force allocation maps loosely to LADs)
lad_total = df.groupby(["lad_code", "month_start"])["incident_count"].transform("sum")
df["lad_month_total"] = lad_total
df["lsoa_share_of_lad"] = np.where(lad_total > 0, df["incident_count"] / lad_total, 0.0)

# Trend / volatility
rm6 = df["rolling_mean_6m"].fillna(0)
std6 = df["rolling_std_6m"].fillna(0)
std12 = df["rolling_std_12m"].fillna(0)
df["cv_6m"] = np.where(rm6 > 0, std6 / rm6, 0.0)
df["cv_12m"] = np.where(rm12 > 0, std12 / rm12, 0.0)
df["trend_3m"] = np.where(
    (rm6 > 0) & (df["rolling_mean_3m"].notna()),
    (df["rolling_mean_3m"].fillna(0) - rm6) / rm6, 0.0
)

print(f"Feature columns: {len(df.columns)}")
df.head()

In [ ]:
# Persist the LSOA feature table
spark_df = spark.createDataFrame(df)
spark_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("varanasi.default.uk_lsoa_features")
spark.sql("ALTER TABLE varanasi.default.uk_lsoa_features SET TBLPROPERTIES ('comment' = 'LSOA panel + ~50 features for CrimeScope UK risk model.')")
spark.sql("OPTIMIZE varanasi.default.uk_lsoa_features ZORDER BY (lsoa_code, month_start)")
print(f"uk_lsoa_features: {len(df):,} rows × {len(df.columns)} cols")

---
## 6. MSOA feature table (rollup)

In [ ]:
msoa_monthly = spark.table("varanasi.default.uk_crime_monthly_msoa").toPandas()
msoa_demo = spark.table("varanasi.default.uk_msoa_demographics").toPandas()
all_msoa = msoa_demo["msoa_code"].unique()
all_months = pd.date_range(msoa_monthly["month_start"].min(), msoa_monthly["month_start"].max(), freq="MS")
idx = pd.MultiIndex.from_product([all_msoa, all_months], names=["msoa_code", "month_start"])
m = pd.DataFrame(index=idx).reset_index()
m["month_start"] = m["month_start"].dt.date
m = m.merge(msoa_monthly, on=["msoa_code", "month_start"], how="left")
for c in ["incident_count", "violent_count", "property_count"]:
    m[c] = m[c].fillna(0).astype(int)
m = m.sort_values(["msoa_code", "month_start"]).reset_index(drop=True)

gM  = m.groupby("msoa_code")["incident_count"]
gMV = m.groupby("msoa_code")["violent_count"]
gMP = m.groupby("msoa_code")["property_count"]
m["y_next_30d_count"] = gM.shift(-1)
m["y_next_30d_violent"] = gMV.shift(-1)
m["y_next_30d_property"] = gMP.shift(-1)
m["y_incidents_12m"] = gM.transform(lambda x: x.rolling(12, min_periods=1).sum())
m = m.dropna(subset=["y_next_30d_count"])
for c in ["y_next_30d_count", "y_next_30d_violent", "y_next_30d_property"]:
    m[c] = m[c].astype(int)

for lag in [1, 2, 3, 6, 12]:
    m[f"lag_{lag}m_count"] = gM.shift(lag)
for period in [3, 6, 12]:
    m[f"rolling_mean_{period}m"] = gM.transform(lambda x: x.shift(1).rolling(period, min_periods=1).mean())
    if period >= 6:
        m[f"rolling_std_{period}m"] = gM.transform(lambda x: x.shift(1).rolling(period, min_periods=1).std())
        m[f"rolling_max_{period}m"] = gM.transform(lambda x: x.shift(1).rolling(period, min_periods=1).max())
        m[f"rolling_min_{period}m"] = gM.transform(lambda x: x.shift(1).rolling(period, min_periods=1).min())
m["mom_change"] = m["incident_count"] - gM.shift(1)
for crime_type, grp in [("violent", gMV), ("property", gMP)]:
    for lag in [1, 3, 6]:
        m[f"{crime_type}_lag_{lag}m"] = grp.shift(lag)
    for period in [3, 6, 12]:
        m[f"{crime_type}_rolling_{period}m"] = grp.transform(
            lambda x: x.shift(1).rolling(period, min_periods=1).mean()
        )
m["violent_ratio"] = np.where(m["incident_count"] > 0, m["violent_count"] / m["incident_count"], 0.0)
m["violent_ratio_6m"] = m.groupby("msoa_code")["violent_ratio"].transform(
    lambda x: x.shift(1).rolling(6, min_periods=1).mean()
)
dt = pd.to_datetime(m["month_start"])
m["month_of_year"] = dt.dt.month
m["month_sin"] = np.sin(2 * math.pi * m["month_of_year"] / 12)
m["month_cos"] = np.cos(2 * math.pi * m["month_of_year"] / 12)
m["year"] = dt.dt.year
m["same_month_last_year"] = gM.shift(12)
m["yoy_change"] = np.where(m["same_month_last_year"].notna(), m["incident_count"] - m["same_month_last_year"], np.nan)
m = m.merge(msoa_demo, on="msoa_code", how="left")
m["log_pop"] = np.where(m["total_pop"] > 0, np.log(m["total_pop"]), 0.0)
rm12 = m["rolling_mean_12m"].fillna(0)
m["crime_rate_per_1k"] = np.where((m["total_pop"] > 0) & (rm12 > 0), rm12 / (m["total_pop"] / 1000.0), 0.0)
m["imd_x_crime"] = m["imd_score"].fillna(0) * rm12
rm6 = m["rolling_mean_6m"].fillna(0)
std6 = m["rolling_std_6m"].fillna(0); std12 = m["rolling_std_12m"].fillna(0)
m["cv_6m"] = np.where(rm6 > 0, std6 / rm6, 0.0)
m["cv_12m"] = np.where(rm12 > 0, std12 / rm12, 0.0)
m["trend_3m"] = np.where(
    (rm6 > 0) & (m["rolling_mean_3m"].notna()),
    (m["rolling_mean_3m"].fillna(0) - rm6) / rm6, 0.0
)

spark.createDataFrame(m).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("varanasi.default.uk_msoa_features")
spark.sql("ALTER TABLE varanasi.default.uk_msoa_features SET TBLPROPERTIES ('comment' = 'MSOA panel + features for CrimeScope UK risk model.')")
spark.sql("OPTIMIZE varanasi.default.uk_msoa_features ZORDER BY (msoa_code, month_start)")
print(f"uk_msoa_features: {len(m):,} rows × {len(m.columns)} cols")